In [81]:
from collections import deque
import torch
from torch import nn, optim
from torch.distributions import Normal
from torch.nn.functional import mse_loss
from tqdm import trange

In [82]:
state_dim = 4
message_dim = 32
action_dim = state_dim
hidden_dim = 64
num_layers = 3

def reward_fn(state, action):
    return 100 - ((state - action) ** 2).sum(dim=-1, keepdim=True)

In [83]:
class MLP(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim, num_layers):
        super(MLP, self).__init__()
        layers = []
        for i in range(num_layers):
            layers.append(nn.Linear(input_dim if i == 0 else hidden_dim, hidden_dim))
            layers.append(nn.LayerNorm(hidden_dim))
            layers.append(nn.LeakyReLU())
        layers.append(nn.Linear(hidden_dim, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, *inputs):
        return self.net(torch.cat(inputs, dim=-1))
    
    def get_normal_policy(self, *inputs):
        mean, log_std = self.forward(*inputs).chunk(2, dim=-1)
        std = log_std.clamp(-20, 2).exp()
        return Normal(mean, std)

In [84]:
message_actor, target_message_actor = (MLP(state_dim, 2 * message_dim, hidden_dim, num_layers) for _ in range(2))
final_actor, target_final_actor = (MLP(message_dim, 2 * action_dim, hidden_dim, num_layers) for _ in range(2))

message_critic, target_message_critic = (MLP(state_dim, 1, hidden_dim, num_layers) for _ in range(2))
final_critic, target_final_critic = (MLP(state_dim + message_dim, 1, hidden_dim, num_layers) for _ in range(2))

message_actor_opt = optim.Adam(message_actor.parameters(), lr=3e-4)
final_actor_opt = optim.Adam(final_actor.parameters(), lr=3e-4)
message_critic_opt = optim.Adam(message_critic.parameters(), lr=3e-4)
final_critic_opt = optim.Adam(final_critic.parameters(), lr=3e-4)

In [85]:
replay_capacity = 10000
num_episodes = 1000
batch_size = 64
epsilon = 0.1
k = 4
lmbda_error = 0.2

In [86]:
# message_rollout_buffer = deque(maxlen=10000)
# final_rollout_buffer = deque(maxlen=10000)
# This isn't needed right now, since the routing is fixed for every sampling.
# We can just rely on using a batch, instead of a replay buffer, to collect samples.
def polyak_update(target, source, tau):
    for target_param, param in zip(target.parameters(), source.parameters()):
        target_param.data.copy_(tau * param.data + (1 - tau) * target_param.data)

In [129]:
for episode in trange(num_episodes):
    # Collect rollout samples
    with torch.no_grad():
        state = torch.randn(k * batch_size, state_dim)
        message_policy = message_actor.get_normal_policy(state)
        message = message_policy.sample()
        message_log_prob = message_policy.log_prob(message).sum(dim=-1, keepdim=True)
        final_value = final_critic(state, message)
        final_policy = final_actor.get_normal_policy(message)
        action = final_policy.sample()
        action_log_prob = final_policy.log_prob(action).sum(dim=-1, keepdim=True)
        final_reward = reward_fn(state, action)
        final_error = (final_value - final_reward).abs()
        message_reward = final_value - lmbda_error * final_error

    # Update critics
    for i in range(k):
        mb_state = state[i * batch_size:(i + 1) * batch_size]
        mb_message = message[i * batch_size:(i + 1) * batch_size]
        mb_message_log_prob = message_log_prob[i * batch_size:(i + 1) * batch_size]
        mb_action = action[i * batch_size:(i + 1) * batch_size]
        mb_action_log_prob = action_log_prob[i * batch_size:(i + 1) * batch_size]
        mb_final_reward = final_reward[i * batch_size:(i + 1) * batch_size]
        mb_message_reward = message_reward[i * batch_size:(i + 1) * batch_size]

        # Update message critic
        message_critic_opt.zero_grad()
        message_critic_output = message_critic(mb_state)
        message_critic_loss = mse_loss(message_critic_output, mb_message_reward)
        message_critic_loss.backward()
        message_critic_opt.step()

        final_critic_opt.zero_grad()
        final_critic_output = final_critic(mb_state, mb_message)
        final_critic_loss = mse_loss(final_critic_output, mb_final_reward)
        final_critic_loss.backward()
        final_critic_opt.step()

        # Update actors
        message_actor_opt.zero_grad()
        message_policy = message_actor.get_normal_policy(mb_state)
        message_new_log_prob = message_policy.log_prob(mb_message).sum(dim=-1, keepdim=True)
        message_prob_ratio = (message_new_log_prob - mb_message_log_prob).exp()
        message_clipped_prob_ratio = message_prob_ratio.clamp(1 - epsilon, 1 + epsilon)
        message_advantage = mb_message_reward - message_critic_output.detach()
        message_advantage = (message_advantage - message_advantage.mean()) / (message_advantage.std() + 1e-8)
        message_actor_loss1 = message_prob_ratio * message_advantage
        message_actor_loss2 = message_clipped_prob_ratio * message_advantage
        message_actor_loss = -torch.min(message_actor_loss1, message_actor_loss2).mean()
        message_actor_loss.backward()
        message_actor_opt.step()

        final_actor_opt.zero_grad()
        final_policy = final_actor.get_normal_policy(mb_message)
        final_new_log_prob = final_policy.log_prob(mb_action).sum(dim=-1, keepdim=True)
        final_prob_ratio = (final_new_log_prob - mb_action_log_prob).exp()
        final_clipped_prob_ratio = final_prob_ratio.clamp(1 - epsilon, 1 + epsilon)
        final_advantage = mb_final_reward - final_critic_output.detach()
        final_advantage = (final_advantage - final_advantage.mean()) / (final_advantage.std() + 1e-8)
        final_actor_loss1 = final_prob_ratio * final_advantage
        final_actor_loss2 = final_clipped_prob_ratio * final_advantage
        final_actor_loss = -torch.min(final_actor_loss1, final_actor_loss2).mean()
        final_actor_loss.backward()
        final_actor_opt.step()

    if episode % 100 == 0:
        print(f"Episode {episode}: Message Critic Loss = {message_critic_loss.item():.4f}, Final Critic Loss = {final_critic_loss.item():.4f}, Message Actor Loss = {message_actor_loss.item():.4f}, Final Actor Loss = {final_actor_loss.item():.4f}, Reward = {final_reward.mean().item():.4f}")

  0%|          | 4/1000 [00:00<01:01, 16.18it/s]

Episode 0: Message Critic Loss = 0.1532, Final Critic Loss = 0.6566, Message Actor Loss = 0.1440, Final Actor Loss = 0.0250, Reward = 97.4882


 10%|█         | 104/1000 [00:06<00:50, 17.87it/s]

Episode 100: Message Critic Loss = 0.0482, Final Critic Loss = 0.0826, Message Actor Loss = 0.1306, Final Actor Loss = 0.2241, Reward = 99.5497


 20%|██        | 204/1000 [00:11<00:44, 18.08it/s]

Episode 200: Message Critic Loss = 0.2626, Final Critic Loss = 0.2045, Message Actor Loss = 0.0793, Final Actor Loss = 0.0553, Reward = 99.1785


... (10 outputs left out)

 80%|████████  | 804/1000 [00:44<00:10, 18.21it/s]

Episode 800: Message Critic Loss = 0.0746, Final Critic Loss = 0.0704, Message Actor Loss = 0.1764, Final Actor Loss = 0.1958, Reward = 99.3849


 85%|████████▌ | 850/1000 [00:47<00:08, 17.90it/s]


KeyboardInterrupt: 

In [134]:
with torch.no_grad():
    state = torch.randn(k * batch_size, state_dim)
    message_policy = message_actor.get_normal_policy(state)
    message = message_policy.sample()
    #message = message.flip(dims=[0])
    message_log_prob = message_policy.log_prob(message).sum(dim=-1, keepdim=True)
    final_value = final_critic(state, message)
    final_policy = final_actor.get_normal_policy(message)
    action = final_policy.sample()
    action_log_prob = final_policy.log_prob(action).sum(dim=-1, keepdim=True)
    final_reward = reward_fn(state, action)
    print(final_reward.mean())
    final_error = (final_value - final_reward).abs()
    message_reward = final_value - lmbda_error * final_error

tensor(99.4872)
